# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulRaffayQureshi/FlyRank-ML-Intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I have selected **Lane 2: Refresh / Content Opportunity Scoring**.

This is formulated as a **Ranking and Scoring** task. Instead of predicting a simple binary class (which forces arbitrary thresholds and ignores operational capacity), we want to output a continuous **Opportunity Score** (from 0 to 100) for every page.

### Why Ranking and Scoring?
In a live business environment, editorial production is strictly constrained by human resources and budget. A content team cannot refresh 5,000 stagnant pages simultaneously. By treating this as a ranking problem, we can sort the entire content catalog from highest-to-lowest opportunity. This allows content managers to confidently slice the top $K$ pages (e.g., the top 50 highest-opportunity candidates) for their weekly writing sprints, maximizing the return on investment (ROI) of their copywriter expenditures.

In [1]:
import pandas as pd
import numpy as np

# Load local starter data to check baseline features
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("--- Describing features that feed our Opportunity Score ---")
print(df[['impressions_90d', 'sessions_90d', 'content_age_days']].describe())

--- Describing features that feed our Opportunity Score ---
       impressions_90d  sessions_90d  content_age_days
count     30000.000000  30000.000000       30000.00000
mean       5200.366300     37.066633         256.16780
std       16838.019547    107.069131         132.70793
min           1.000000      1.000000          90.00000
25%          81.000000      2.000000         132.00000
50%         731.000000      7.000000         236.00000
75%        3615.250000     27.000000         333.00000
max      517715.000000   4345.000000         564.00000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### The Target Formulation
Rather than relying on subjective, human-assigned flags or a defined rule, our model will utilize a **future-looking observed target proxy** calculated purely from historical performance data:

* **The Proxy:** We will predict whether a given page achieves a $\ge 15\%$ uplift in organic search impressions (or clicks) in a defined future performance window (e.g., Day $+1$ to Day $+30$) following a target threshold.
* **Source of the Label:** This is an **observed outcome** generated from actual Google Search Console performance tracking data.
* **Strict Temporal Discipline:** To avoid data leakage, we will strictly separate our feature inputs and target labels. All features must be constructed from historical data (Day $-90$ to Day $0$), while the label is computed strictly in the future outcome window (Day $+1$ to Day $+30$).

In [2]:
# Mocking a future 15% traffic uplift outcome relative to past 90-day impressions
np.random.seed(42)
df['mock_future_impressions'] = df['impressions_90d'] * np.random.uniform(0.8, 1.3, size=len(df))
df['target_recovered'] = (df['mock_future_impressions'] >= df['impressions_90d'] * 1.15).astype(int)

print("Target label class distribution (1 = Recovered, 0 = Stagnant):")
print(df['target_recovered'].value_counts(normalize=True))

Target label class distribution (1 = Recovered, 0 = Stagnant):
target_recovered
0    0.7029
1    0.2971
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Primary Metric: Precision @ K (specifically, Precision@50)

Because this model serves as a decision-support framework, the ultimate validation is: *"If the system recommends the top 50 pages to refresh, how many of those pages actually achieve a traffic recovery?"*

* **Why Precision @ K?** Content strategy teams are bound by weekly resource constraints (e.g., they can only write 50 updates per week). They do not care about maximizing "Recall" (finding every single declining page on the website). They care about the high precision of the short queue they are assigned to work on. If Precision@50 is $0.70$, then 35 of the 50 updated pages recover, keeping operational waste low.
* **Secondary Evaluation Metric:** Precision-Recall Area Under the Curve (PR-AUC), which is far more stable than ROC-AUC for imbalanced ranking distributions.

In [3]:
# Let's simulate a scoring run, sort by score, and measure Precision@50
df['mock_prediction_score'] = np.random.uniform(0, 1, size=len(df))

# Take top 50 predicted opportunities
top_50 = df.sort_values(by='mock_prediction_score', ascending=False).head(50)
precision_at_50 = top_50['target_recovered'].mean()

print(f"Simulated Precision@50: {precision_at_50:.2%}")

Simulated Precision@50: 26.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### The Unit of Analysis (Grain)
* **The Grain:** One row in our data slice represents **one unique content item** (`content_id`) associated with a **specific client** (`client_id`) evaluated at a single point in time. 
* **The Features:** Every row tracks its historical performance footprint (impressions, sessions) and metadata attributes (content age in days).

In [4]:
# Show the exact grain of the dataset with shape confirmation
unit_df = df[['client_id', 'content_id', 'impressions_90d', 'sessions_90d', 'content_age_days']].copy()

print(f"Total Rows (Sample Footprint): {len(unit_df):,}")
print("\nFirst 5 rows of our Unit of Analysis dataframe:")
print(unit_df.head(5).to_string(index=False))

Total Rows (Sample Footprint): 30,000

First 5 rows of our Unit of Analysis dataframe:
        client_id           content_id  impressions_90d  sessions_90d  content_age_days
client_f369cb89fc content_304f48230142             3803            17               187
client_4e07408562 content_a1fb4e703a9e            15320             9               445
client_7f2253d7e2 content_9aa793d4d895            12581            11               141
client_19581e27de content_331d6c4de07b            11751            78               463
client_3fdba35f04 content_d99b7a2d90ca            19140           145               263


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Beats Brittle Static Rules
A typical heuristic-based rule like `if content_age_days > 180 AND impressions_90d > 1000 then refresh` fails in production due to the highly non-linear dynamics of search indexing:

1. **Non-linear Rank Decay:** A loss of position from Rank 2 to Rank 6 results in a catastrophic drop in click volume, whereas a drop from Rank 12 to Rank 16 is mathematically negligible. Standard rules cannot easily handle non-linear scaling.
2. **The Search Volume Disconnect:** As shown in our initial data exploration, search volume does not scale linearly with actual impressions.
3. **Multi-variable Interactions:** Opportunities are hidden in the multi-dimensional combinations of click-through rate (CTR) decay, historical position tiers, content age, and user engagement. Hand-crafting nested `if-else` blocks to capture these dynamic thresholds is brittle, hard to maintain, and highly error-prone. ML excels at discovering these complex multi-variable decision boundaries automatically.

In [5]:
# Show mathematical proof that simple variables don't correlate linearly
correlation = df['content_age_days'].corr(df['impressions_90d'])
print(f"Correlation between Content Age (Days) and 90d Impressions: {correlation:.4f}")
print("Because this correlation is near zero, a simple 'refresh if older than X days' rule is highly inefficient.")

Correlation between Content Age (Days) and 90d Impressions: -0.0007
Because this correlation is near zero, a simple 'refresh if older than X days' rule is highly inefficient.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.